now decision whether dataset should remain in JSON format or CSV,
I think good to convert json into CSV

first lets try to load 1 json file as pandas dataframe


In [ ]:
import pandas as pd
pd.read_json('contest_496/leaderboard/leaderboard_1.json') #is giving error
# ValueError: All arrays must be of the same length

will convert everything to CSV
first processing contest 496 as most files I am scraping it all fetched


In [4]:
import pandas as pd
df = pd.read_csv('test.csv')
df.head()

,contest_id,username,rank,score,total_time,total_fail,lang_count,country,profile_rank,total_solved,easy,medium,hard,reputation,postview,global_rank,first_rating,last_rating,Active_Time
0,1,VeritasVelata,1,20,338,0,1,Sanctuary,19,3893,935,2037,921,137,18693,25,1885.526,3321.441,147.5
1,1,ayushkumar980,2,20,284,0,1,India,1433822,106,23,53,30,0,4,77,1810.747,3109.444,98.0
2,1,raunak111,3,20,297,0,1,India,341067,358,121,196,41,0,7,99577,1425.508,1721.921,602.0
3,1,Mona0405,4,20,213,0,1,NaN,1557154,95,24,48,23,0,0,15295,2058.392,2058.392,0.0
4,1,scale5558,5,20,210,0,1,NaN,112045,628,168,338,122,4,0,4752,1560.748,2283.780,189.5


missing contest submission stats. from leaderboard.json.

need to encode the user contest history from into single columns

first try to make 1 row of valid CSV then try to extend it to make single CSV file of 10000 rows of all users

need to flatted nested JSON into 1 valid row. 
https://chatgpt.com/s/t_69da738d431481918e1dd7e8e4ad0162


In [5]:
import json
CONTEST=str(496)
leaderboard=open('contest_496/leaderboard/leaderboard_1.json','r')
ld=json.load(leaderboard)
ld['user_num']


39352

NOW all data for a particular user is is below
Rank 1 

In [6]:
ld['total_rank'][0] #rank 1 profile basics (contest data)

{'contest_id': 1283,
 'username': 'Veritas Velata',
 'username_color': None,
 'user_badge': {'icon': '/static/images/badges/guardian.png',
  'display_name': 'Guardian'},
 'user_slug': 'VeritasVelata',
 'country_code': '',
 'country_name': 'Sanctuary',
 'rank': 1,
 'score': 20,
 'finish_time': 1775356616,
 'global_ranking': 0,
 'data_region': 'US',
 'avatar_url': 'https://assets.leetcode.com/users/VeritasVelata/avatar_1773462219.png',
 'replays': None}

In [7]:
ld['submissions'][0] #rank 1 current contest performance

{'4149': {'id': 30801559,
  'date': 1775356478,
  'question_id': 4149,
  'submission_id': 1969080592,
  'status': 10,
  'contest_id': 1283,
  'data_region': 'US',
  'fail_count': 0,
  'lang': 'csharp'},
 '4263': {'id': 30803015,
  'date': 1775356616,
  'question_id': 4263,
  'submission_id': 1969082773,
  'status': 10,
  'contest_id': 1283,
  'data_region': 'US',
  'fail_count': 0,
  'lang': 'csharp'},
 '4268': {'id': 30801069,
  'date': 1775356366,
  'question_id': 4268,
  'submission_id': 1969079506,
  'status': 10,
  'contest_id': 1283,
  'data_region': 'US',
  'fail_count': 0,
  'lang': 'csharp'},
 '4273': {'id': 30800953,
  'date': 1775356278,
  'question_id': 4273,
  'submission_id': 1969078872,
  'status': 10,
  'contest_id': 1283,
  'data_region': 'US',
  'fail_count': 0,
  'lang': 'csharp'}}

above quesstions are not ordered need to find Q1 using questionss in leaderboard and then arrangee them for writing like
Q1 Q2 Q3 Q4 time


In [9]:
Q1_ID=ld['questions'][0]['question_id'] #credit 4
Q2_ID=ld['questions'][1]['question_id'] #credit 5
Q3_ID=ld['questions'][2]['question_id'] #credit 5
Q4_ID=ld['questions'][3]['question_id'] #creadit 6 

Q1_ID

4273

now use this ID to identify the Q1

In [111]:
ld['submissions'][0][str(Q1_ID)] #prints only Q1 data now extract time


KeyError: 4273

it is possible not all Qid is accessible in submissions so safely handle it using .get()
print(d.get("age", 0))
or try except is more better
try:
    print(d["age"])
except KeyError:
    print("Key does not exist")

In [14]:

#not safe to check
#it 
#safe way to check 
# qi=ld['submissions'][0].get(qid[i],0), q1_time=qi['date']

q1_time=ld['submissions'][0][str(Q1_ID)]['date'] #Q1 time
q2_time=ld['submissions'][0][str(Q2_ID)]['date'] #Q2 time
q3_time=ld['submissions'][0][str(Q3_ID)]['date'] #Q3 time
q4_time=ld['submissions'][0][str(Q4_ID)]['date'] #Q4 time
print(q1_time,end=' ')
print(q2_time,end=' ')
print(q3_time,end=' ')
print(q4_time,end=' ')


1775356278 1775356366 1775356478 1775356616 

this is EPOCH time format
can use pandas to transform whole column like this into this 
df['date'] = pd.to_datetime(df['date'], unit='s')


In [16]:

q1_fail_count=ld['submissions'][0][str(Q4_ID)]['fail_count'] #Q1 fail count
q1_fail_count

0

total problem solve count
Status  represent whether question accepted or not. can use it to determine number of problems solved for curr contest
status =10 means accepted

In [20]:
#function to find total problem count
qid=[str(Q1_ID),str(Q2_ID),str(Q3_ID),str(Q4_ID)]
total_solved=0
for i in range(0,len(qid)):
    #it is possible that qid question never submits so no data so first check whther exits or not then access it
    #can use try except also more genreal way
    code=ld['submissions'][0].get(qid[i],0)
    if(code['status']==10):
        total_solved+=1

print(total_solved) #use this total solved as 1 column for that current user


4


Row headers from leaderboard file

Rank (-1) |user_slug ("") | score | num_qs in curr contest (0)| finishTime | Q1timeStamp (00)| Q2TimeStamp (00) | Q3TimeStamp (00)| Q4TimeStamp (00) | Q1_fail_Count (-1)| Q2_fail_count (-1)| Q3_fail_count(-1) | Q4_fail_count (-1) | UserName | countryname ("") 

total ->15cols from here


will provide this this outside csv file in txt file
Contest INFO
contest-496 important properites -> 
Q1 - 4 points
Q2 - 5 points
Q3 - 5 points
Q4 - 6 points
so max score possible = 20
total users given this contest = ld['usernum']

total 15 scores are possible

In [ ]:
pgFile=open('contest_496/user/pg_1/0.json','r') # 0 here represents the first on current pg
pg=json.load(pgFile)['data'] #data will exist so use here only


In [30]:
print(pg)

{'matchedUser': {'username': 'VeritasVelata', 'profile': {'ranking': 19, 'realName': 'Veritas Velata', 'countryName': 'Sanctuary', 'postViewCount': 18693, 'reputation': 137}, 'submitStats': {'acSubmissionNum': [{'difficulty': 'All', 'count': 3893, 'submissions': 4873}, {'difficulty': 'Easy', 'count': 935, 'submissions': 1212}, {'difficulty': 'Medium', 'count': 2037, 'submissions': 2589}, {'difficulty': 'Hard', 'count': 921, 'submissions': 1072}]}}, 'userContestRanking': {'attendedContestsCount': 23, 'rating': 3321.4411190574583, 'globalRanking': 25, 'topPercentage': 0.01}, 'userContestRankingHistory': [{'attended': True, 'trendDirection': 'UP', 'problemsSolved': 4, 'ranking': 52, 'rating': 1885.526, 'contest': {'title': 'Biweekly Contest 169', 'startTime': 1762612200}}, {'attended': True, 'trendDirection': 'DOWN', 'problemsSolved': 0, 'ranking': 22660, 'rating': 1785.783, 'contest': {'title': 'Weekly Contest 476', 'startTime': 1763260200}}, {'attended': True, 'trendDirection': 'UP', 'p

In [34]:
#for data inside profile
pg['matchedUser']['profile']

{'ranking': 19,
 'realName': 'Veritas Velata',
 'countryName': 'Sanctuary',
 'postViewCount': 18693,
 'reputation': 137}

In [38]:
# for problem counts

pg['matchedUser']['submitStats']['acSubmissionNum'][0] #will give all Qs solved data

{'difficulty': 'All', 'count': 3893, 'submissions': 4873}

In [39]:
 #will give all Qs solved data --> easy medium hard
print(pg['matchedUser']['submitStats']['acSubmissionNum'][1]) #easy
print(pg['matchedUser']['submitStats']['acSubmissionNum'][2]) #medium
print(pg['matchedUser']['submitStats']['acSubmissionNum'][3]) #hard

{'difficulty': 'Easy', 'count': 935, 'submissions': 1212}
{'difficulty': 'Medium', 'count': 2037, 'submissions': 2589}
{'difficulty': 'Hard', 'count': 921, 'submissions': 1072}


Row headers from corresponding pg file pg_{leaderboardfileNO}. these columns will follow the abovee cols

 ...| rankingP |postViewCount (0) |reputation (0) | ALL_QS_SOLVED (0)|TOTAL_QS_Sub (0)|TOTAL_QS_Sub (0) | EASY_QS_SOLVED (0) | Easy_QS_Sub(0) | MED_QS_SOLVED (0) |  MED_QS_Sub(0) |HARD_QS_SOLVED (0) |HARD_QS_Sub (0) |
    
    |ContestCount | curr_rating | globalRanking |top percentage | ContestHistoryString 

    pg cols = 16

    total cols=16+14 = 31 featrues so

    final ideally final dataset should be 10,000 x 30

    https://chatgpt.com/s/t_69daa0be2a948191ad6c740ebb19c12b

In [40]:
pg['userContestRanking'] #all contest info


{'attendedContestsCount': 23,
 'rating': 3321.4411190574583,
 'globalRanking': 25,
 'topPercentage': 0.01}

In [42]:
#right now not encodeing user contest history will stringgy the json value so that able to store in csv file
userContestHistoryString=json.dumps(pg['userContestRankingHistory'])
print(userContestHistoryString)
#will encode in this in future during ML problem

[{"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 52, "rating": 1885.526, "contest": {"title": "Biweekly Contest 169", "startTime": 1762612200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 22660, "rating": 1785.783, "contest": {"title": "Weekly Contest 476", "startTime": 1763260200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 58, "rating": 1968.706, "contest": {"title": "Weekly Contest 481", "startTime": 1766284200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 233, "rating": 2056.281, "contest": {"title": "Weekly Contest 482", "startTime": 1766889000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 239, "rating": 2126.35, "contest": {"title": "Weekly Contest 484", "startTime": 1768098600}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 626, "rating": 2151.104, "contest": {"title": "Biweekly Contest 174", "

In [87]:
#now safe versions
USER=0 #curr page has 0t o 24 users possible
rank=-1
#curr page 0 to 24 person , right now curr pg, 1st person
d1=ld['total_rank'][USER]



#if rank is null then will then will remove those using pandas
#1
#check total rank always exist, user 0 always exist , rank always exist, can be null value
rank=d1['rank']

#2
#user_slug always exist as key value is also present
userSlug=d1['user_slug']

#15
userName=d1['username']
#3
#key exist always value can ve null
countryName=d1['country_name']  

#4
#score key also always exist. 
score=d1['score']

#5
#finish time of contest
finishTime=d1['finish_time']

#for 6 to 13 cols
#qid is static for current contest so same for all make it global
d1=ld['submissions'][0]
arr=[None]*8
problemSolved=0 #problem count in current
for i in range(0,4):
    q_time=None
    q_fail_cnt=None
    try :
        q_time=d1[qid[i]]['date'] 
        q_fail_cnt=d1[qid[i]]['fail_count']
        if(d1[qid[i]]['status'] ==10):
            problemSolved+=1
    except :
        q_time=None
        q_fail_cnt=None

    arr[i]=q_time
    arr[4+i]=q_fail_cnt

#6
# print(problemSolved)

#7,8,9,10,11,12,13,14
# print(arr)

currRow=[rank,userSlug,score,problemSolved,finishTime]
for i in range(0,8):
    currRow.append(arr[i])

currRow.append(userName)
currRow.append(countryName)


pgFile=open(f'contest_496/user/pg_1/{USER}.json','r')
pg=json.load(pgFile)['data']
arr2=[None]*15


#now data from pg file
d1=pg['matchedUser']['profile']
currRow.append(d1['ranking']) #profile ranking ranking P
currRow.append(d1['postViewCount'])
currRow.append(d1['reputation'])

#problem counts
d1=pg['matchedUser']['submitStats']['acSubmissionNum']
for i in range(0,4):
    currRow.append(d1[i]['count'])
    currRow.append(d1[i]['submissions'])

#now contestRankingInfo

#check can be null or not
d1=pg['userContestRanking'] #if someone participated in contest then He will have this column
currRow.append(d1['attendedContestsCount'])
currRow.append(d1['rating'])
currRow.append(d1['globalRanking'])
currRow.append(d1['topPercentage'])

d1=pg['userContestRankingHistory']
#userHistor
userHistory=json.dumps(d1)
#1feature history of allcontests of user 
#need to encode this in a better way
currRow.append(userHistory)
print(f'total Features : {len(currRow)}')
for i in range(0,len(currRow)):
    print(f" {i} : {currRow[i]}")

total Features : 31
 0 : 1
 1 : VeritasVelata
 2 : 20
 3 : 4
 4 : 1775356616
 5 : 1775356278
 6 : 1775356366
 7 : 1775356478
 8 : 1775356616
 9 : 0
 10 : 0
 11 : 0
 12 : 0
 13 : Veritas Velata
 14 : Sanctuary
 15 : 19
 16 : 18693
 17 : 137
 18 : 3893
 19 : 4873
 20 : 935
 21 : 1212
 22 : 2037
 23 : 2589
 24 : 921
 25 : 1072
 26 : 23
 27 : 3321.4411190574583
 28 : 25
 29 : 0.01
 30 : [{"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 52, "rating": 1885.526, "contest": {"title": "Biweekly Contest 169", "startTime": 1762612200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 22660, "rating": 1785.783, "contest": {"title": "Weekly Contest 476", "startTime": 1763260200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 58, "rating": 1968.706, "contest": {"title": "Weekly Contest 481", "startTime": 1766284200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 233, "rating": 2056.281,

In [ ]:
def makeRow(user):
    rank=-1
    #curr page 0 to 24 person , right now curr pg, 1st person
    d1=ld['total_rank'][user]



    #if rank is null then will then will remove those using pandas
    #1
    #check total rank always exist, user 0 always exist , rank always exist, can be null value
    rank=d1['rank']

    #2
    #user_slug always exist as key value is also present
    userSlug=d1['user_slug']

    #15
    userName=d1['username']
    #3
    #key exist always value can ve null
    countryName=d1['country_name']  

    #4
    #score key also always exist. 
    score=d1['score']

    #5
    #finish time of contest
    finishTime=d1['finish_time']

    #for 6 to 13 cols
    #qid is static for current contest so same for all make it global
    d1=ld['submissions'][0]
    arr=[None]*8
    problemSolved=0 #problem count in current
    for i in range(0,4):
        q_time=None
        q_fail_cnt=None
        try :
            q_time=d1[qid[i]]['date'] 
            q_fail_cnt=d1[qid[i]]['fail_count']
            if(d1[qid[i]]['status'] ==10):
                problemSolved+=1
        except :
            q_time=None
            q_fail_cnt=None

        arr[i]=q_time
        arr[4+i]=q_fail_cnt

    #6
    # print(problemSolved)

    #7,8,9,10,11,12,13,14
    # print(arr)

    currRow=[rank,userSlug,score,problemSolved,finishTime]
    for i in range(0,8):
        currRow.append(arr[i])

    currRow.append(userName)
    currRow.append(countryName)


    pgFile=open(f'contest_496/user/pg_1/{USER}.json','r')
    pg=json.load(pgFile)['data']
    arr2=[None]*15


    #now data from pg file
    d1=pg['matchedUser']['profile']
    currRow.append(d1['ranking']) #profile ranking ranking P
    currRow.append(d1['postViewCount'])
    currRow.append(d1['reputation'])

    #problem counts
    d1=pg['matchedUser']['submitStats']['acSubmissionNum']
    for i in range(0,4):
        currRow.append(d1[i]['count'])
        currRow.append(d1[i]['submissions'])

    #now contestRankingInfo

    #check can be null or not
    d1=pg['userContestRanking'] #if someone participated in contest then He will have this column
    currRow.append(d1['attendedContestsCount'])
    currRow.append(d1['rating'])
    currRow.append(d1['globalRanking'])
    currRow.append(d1['topPercentage'])

    d1=pg['userContestRankingHistory']
    #userHistor
    userHistory=json.dumps(d1)
    #1feature history of allcontests of user 
    #need to encode this in a better way
    currRow.append(userHistory)
    return currRow

In [98]:
makeRow(5)

[6,
 'user8376lk',
 20,
 4,
 1775356831,
 1775356278,
 1775356366,
 1775356478,
 1775356616,
 0,
 0,
 0,
 0,
 'Harsh Shastri',
 None,
 19,
 18693,
 137,
 3893,
 4873,
 935,
 1212,
 2037,
 2589,
 921,
 1072,
 23,
 3321.4411190574583,
 25,
 0.01,
 '[{"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 52, "rating": 1885.526, "contest": {"title": "Biweekly Contest 169", "startTime": 1762612200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 22660, "rating": 1785.783, "contest": {"title": "Weekly Contest 476", "startTime": 1763260200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 58, "rating": 1968.706, "contest": {"title": "Weekly Contest 481", "startTime": 1766284200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 233, "rating": 2056.281, "contest": {"title": "Weekly Contest 482", "startTime": 1766889000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ran

In [99]:
makeRow(6)

[7,
 'rakkalra',
 20,
 4,
 1775356887,
 1775356278,
 1775356366,
 1775356478,
 1775356616,
 0,
 0,
 0,
 0,
 'R Kalra',
 None,
 19,
 18693,
 137,
 3893,
 4873,
 935,
 1212,
 2037,
 2589,
 921,
 1072,
 23,
 3321.4411190574583,
 25,
 0.01,
 '[{"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 52, "rating": 1885.526, "contest": {"title": "Biweekly Contest 169", "startTime": 1762612200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 22660, "rating": 1785.783, "contest": {"title": "Weekly Contest 476", "startTime": 1763260200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 58, "rating": 1968.706, "contest": {"title": "Weekly Contest 481", "startTime": 1766284200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 233, "rating": 2056.281, "contest": {"title": "Weekly Contest 482", "startTime": 1766889000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 4, "ranking": 2

In [105]:
import csv

f=open('testing.csv','w')
writer=csv.writer(f)

#header row
writer.writerow([
    'Rank',
    'User_slug',
    'Score',
    'CurrConSolved',
    'FinishTimeStamp',
    'Q1TimeStamp',
    'Q2TimeStamp',
    'Q3TimeStamp',
    'Q4TimeStamp',
    'Q1FailCnt',
    'Q2FailCnt',
    'Q3FailCnt',
    'Q4FailCnt',
    'UserName',
    'CountryName',
    'RankingP',
    'PostViewCnt',
    'Reputation',
    'TotalQsAcc',
    'TotalQsSub',
    'EasyQsAcc',
    'EasyQsSub',
    'MedQsAcc',
    'MedQsSub',
    'HardQsAcc',
    'HardQsSub',
    'ContestCnt',
    'CurrRating',
    'GlobalRank',
    'topPercentage',
    'ContestHistory'
    ]
)
# writer.writerow(makeRow(5))
for i in range(0,24+1):
    writer.writerow(makeRow(i))


In [110]:
name='leaderboard_90.json'
print(name)
print(len(name))
print(name[12:]) # 90.json
print(name[12:-5]) #remove last 5 chars which is .json


leaderboard_90.json
19
90.json
90


In [ ]:
#some usernames are with wierd characters can we parse them with utf-8 encodeing
#data hasa come from graphQl queries for wierd usernames
#analysing can we make row for special characters username data has come from API
#pg10 user 19
import mapping
ld_utf_8=open('contest_496/leaderboard/leaderboard_10.json','r',encoding='utf-8')
pg_utf_8=open('contest_496/user/pg_10/19.json','r',encoding='utf-8')
ld=json.load(ld_utf_8)
pg=json.load(pg_utf_8)

#not able to process unique usernames with special chars
usrRow=mapping.makeRow(ld,pg,19,qid)
print(usrRow)






KeyError: 'matchedUser'

currently not able to process uniques usernames